<div align="center">

<a href="https://colab.research.google.com/github/utkukose/machine-ethics-ai-safety-NB-lecture/blob/main/exams/final/FIN2_explanations_under_scrutiny.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Final Capstone: Building and Auditing a Safer AI System

## Project 2: Explanations under scrutiny: faithfulness, stability and manipulation

**Machine Ethics and Artificial Intelligence Safety (11118BLG001)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## The project

Three local attribution methods are implemented for a gradient boosting classifier and evaluated for faithfulness with deletion curves, for agreement, for stability under small input noise and with a model randomisation sanity check [1, 2, 3]. A final experiment shows how a scaffolded model can hide its reliance on a sensitive attribute from perturbation-based explanations [4].

**Required extensions.** Add at least one further attribution method, for example GEMEX or integrated gradients on a differentiable model [5], and one further faithfulness metric. Test whether a defence, such as on-manifold perturbations, restores the hidden attribute in the scaffolding experiment, and discuss what the results imply for audits that rely on explanations [6].

## What this example implements

The notebook implements occlusion, a LIME-style local surrogate and Shapley value sampling, computes deletion faithfulness, Kendall rank agreement, stability and a sanity check, and reproduces the logic of a scaffolding attack on a synthetic dataset.

Run the cells in order. The example is a baseline: the capstone extends it, evaluates the extensions and reports the results.

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. Three attribution methods and faithfulness

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy.stats import kendalltau, spearmanr

data = load_breast_cancer()
X, y = data.data, 1 - data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=SEED, stratify=y)
sc = StandardScaler().fit(X_tr)
Z_tr, Z_te = sc.transform(X_tr), sc.transform(X_te)
gb = GradientBoostingClassifier(random_state=SEED).fit(Z_tr, y_tr)
f = lambda Z: gb.predict_proba(Z)[:, 1]
D = Z_tr.shape[1]
baseline = np.zeros(D)                        # the training mean in standardised units

def occlusion(x, f=f):
    Zs = np.tile(x, (D, 1)); Zs[np.arange(D), np.arange(D)] = 0.0
    return f(x[None])[0] - f(Zs)

def lime_like(x, f=f, n=1500, seed=SEED):
    r = np.random.default_rng(seed)
    Zs = x + r.normal(0, 1, (n, D))
    wts = np.exp(-np.linalg.norm(Zs - x, axis=1) ** 2 / (2 * (1.5 * np.sqrt(D)) ** 2))
    return Ridge(alpha=1.0).fit(Zs - x, f(Zs), sample_weight=wts).coef_

def shapley_sampling(x, f=f, n_perm=60, seed=SEED):
    r, phi = np.random.default_rng(seed), np.zeros(D)
    for _ in range(n_perm):
        order = r.permutation(D)
        Zs = np.tile(baseline, (D + 1, 1))
        for k, j in enumerate(order):
            Zs[k + 1:, j] = x[j]
        phi[order] += np.diff(f(Zs))
    return phi / n_perm

def deletion_auc(x, attr, f=f):
    Zs = np.tile(x, (D + 1, 1))
    for k, j in enumerate(np.argsort(-np.abs(attr))):
        Zs[k + 1:, j] = 0.0
    v = f(Zs)
    curve = v if v[0] >= 0.5 else 1 - v
    return float((curve[:-1] + curve[1:]).sum() / 2 / D)

METHODS = {"occlusion": occlusion, "LIME-style surrogate": lime_like, "Shapley sampling": shapley_sampling,
           "random": lambda x: np.random.default_rng(SEED).normal(size=D)}
idx = np.arange(20)
attrs = {m: np.array([fn(Z_te[i]) for i in idx]) for m, fn in METHODS.items()}
faith = {m: np.mean([deletion_auc(Z_te[i], a) for i, a in zip(idx, attrs[m])]) for m in METHODS}
print("deletion AUC, lower means more faithful:", {m: round(v, 3) for m, v in faith.items()})

## 2. Agreement, stability and a sanity check

In [ ]:
names = [m for m in METHODS if m != "random"]
tau = {(a, b): np.mean([kendalltau(np.abs(attrs[a][k]), np.abs(attrs[b][k]))[0] for k in range(len(idx))]) for a in names for b in names if a < b}
print("rank agreement (Kendall tau):", {f"{a} vs {b}": round(v, 2) for (a, b), v in tau.items()})
rng = np.random.default_rng(SEED)
for m in names:
    ch = []
    for k, i in enumerate(idx[:10]):
        a0 = attrs[m][k]; a1 = METHODS[m](Z_te[i] + rng.normal(0, 0.05, D))
        ch.append(np.linalg.norm(a1 - a0) / (np.linalg.norm(a0) + 1e-12))
    print(f"{m}: mean relative change under small input noise {np.mean(ch):.3f}")
y_shuffled = np.random.default_rng(SEED).permutation(y_tr)
gb_rand = GradientBoostingClassifier(random_state=SEED).fit(Z_tr, y_shuffled)
f_rand = lambda Z: gb_rand.predict_proba(Z)[:, 1]
for m in names:
    rho = np.mean([spearmanr(np.abs(METHODS[m](Z_te[i])), np.abs(METHODS[m](Z_te[i], f=f_rand)))[0] for i in idx[:8]])
    print(f"sanity check, {m}: rank correlation with a model trained on shuffled labels {rho:.2f}")

## 3. Hiding a sensitive attribute from explanations

In [ ]:
rng = np.random.default_rng(SEED)
m = 3000
Xf = np.c_[rng.binomial(1, 0.5, m).astype(float), rng.normal(0, 1, m), rng.normal(0, 1, m)]   # sensitive attribute, two innocuous features
biased = lambda Z: (Z[:, 0] > 0.5).astype(float)
innocuous = lambda Z: (Z[:, 1] > 0).astype(float)
pert = Xf[rng.integers(0, m, m)] + rng.normal(0, 1, (m, 3))
det = RandomForestClassifier(n_estimators=100, random_state=SEED).fit(np.r_[Xf, pert], np.r_[np.ones(m), np.zeros(m)])

def scaffold(Z):
    return np.where(det.predict_proba(Z)[:, 1] >= 0.5, biased(Z), innocuous(Z))

def lime_generic(x, fn, n=1000, seed=SEED):
    r = np.random.default_rng(seed)
    Zs = x + r.normal(0, 1, (n, len(x)))
    return Ridge(alpha=1.0).fit(Zs - x, fn(Zs)).coef_

coefs = np.abs(np.array([lime_generic(Xf[i], scaffold) for i in range(30)])).mean(0)
print("share of real decisions that follow the sensitive attribute:", np.mean(scaffold(Xf) == biased(Xf)))
print("mean |surrogate coefficient|: sensitive", coefs[0].round(3), " innocuous 1", coefs[1].round(3), " innocuous 2", coefs[2].round(3))
try:
    from gemex import Explainer, GemexConfig
    print("GEMEX is installed and can be added as a further attribution method for comparison.")
except ImportError:
    print("GEMEX is not installed; pip install gemex adds a geometric attribution method for comparison.")

## Report and submission

The technical report states the question, explains the design choices, presents the experiments with the figures produced by the notebook and discusses the ethical and safety implications and the limitations of the results. It cites at least eight sources in square brackets, at least four of them from the course readings, and links to the final Colab notebook.

This capstone also supports self-learning and can be completed at any pace. When the course is taught actively in a semester, the final capstone is sent by e-mail to utkukose@sdu.edu.tr or utkukose@gmail.com no later than 23:53 (Türkiye time) on the last Sunday of Week 14, with the report and all code files attached or linked.

## References

[1] Ribeiro, M. T., Singh, S., & Guestrin, C. (2016). "Why should I trust you?": Explaining the predictions of any classifier. In *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining* (pp. 1135-1144). <https://doi.org/10.1145/2939672.2939778>

[2] Lundberg, S. M., & Lee, S.-I. (2017). A unified approach to interpreting model predictions. In *Advances in Neural Information Processing Systems 30*.

[3] Adebayo, J., Gilmer, J., Muelly, M., Goodfellow, I., Hardt, M., & Kim, B. (2018). Sanity checks for saliency maps. In *Advances in Neural Information Processing Systems 31*.

[4] Slack, D., Hilgard, S., Jia, E., Singh, S., & Lakkaraju, H. (2020). Fooling LIME and SHAP: Adversarial attacks on post hoc explanation methods. In *Proceedings of the AAAI/ACM Conference on AI, Ethics, and Society* (pp. 180-186). <https://doi.org/10.1145/3375627.3375830>

[5] Kose, U. (2026). *GEMEX: Geodesic Entropic Manifold Explainability (Version 1.2.2) [Python package]*. Python Package Index. <https://pypi.org/project/gemex/>

[6] Rudin, C. (2019). Stop explaining black box machine learning models for high stakes decisions and use interpretable models instead. *Nature Machine Intelligence*, 1(5), 206-215. <https://doi.org/10.1038/s42256-019-0048-x>